In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, KFold
import gc

# Set plotting style
plt.style.use('fivethirtyeight')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully.")


In [ ]:
# Configuration
base_path = 'datathon-2026-round-1/'

print("Loading datasets...")
try:
    orders = pd.read_csv(f'{base_path}orders.csv', parse_dates=['order_date'])
    products = pd.read_csv(f'{base_path}products.csv')
    returns = pd.read_csv(f'{base_path}returns.csv')
    web_traffic = pd.read_csv(f'{base_path}web_traffic.csv', parse_dates=['date'])
    order_items = pd.read_csv(f'{base_path}order_items.csv')
    customers = pd.read_csv(f'{base_path}customers.csv')
    geography = pd.read_csv(f'{base_path}geography.csv')
    sales = pd.read_csv(f'{base_path}sales.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
    payments = pd.read_csv(f'{base_path}payments.csv')
    promotions = pd.read_csv(f'{base_path}promotions.csv', parse_dates=['start_date', 'end_date'])
    inventory = pd.read_csv(f'{base_path}inventory.csv', parse_dates=['snapshot_date'])
    sample_sub = pd.read_csv(f'{base_path}sample_submission.csv', parse_dates=['Date'])
    print("All data loaded successfully.")
except Exception as e:
    print("Error:", e)


In [ ]:
# Web Traffic Feature Preparation
wt = web_traffic.groupby('date').agg({
    'sessions':'sum',
    'page_views':'sum',
    'unique_visitors':'sum',
    'bounce_rate':'mean',
    'avg_session_duration_sec':'mean'
}).reset_index().rename(columns={'date':'Date'})

full_dr = pd.date_range(sales.Date.min(), sample_sub.Date.max(), freq='D')
wt = wt.set_index('Date').reindex(full_dr).ffill().bfill().reset_index().rename(columns={'index':'Date'})

train_df = sales.copy()
train_df['month'] = train_df.Date.dt.month
train_df['dow'] = train_df.Date.dt.dayofweek
train_df['day'] = train_df.Date.dt.day

target_enc_month = train_df.groupby('month')['Revenue'].mean().to_dict()
target_enc_dow = train_df.groupby('dow')['Revenue'].mean().to_dict()
target_enc_day = train_df.groupby('day')['Revenue'].mean().to_dict()


In [ ]:
def get_promo_features(dates):
    df = pd.DataFrame({'Date': dates})
    y, m, d = df.Date.dt.year, df.Date.dt.month, df.Date.dt.day
    df['in_spring'] = (((m==3)&(d>=18))|((m==4)&(d<=17))).astype(int)
    df['in_midyear'] = (((m==6)&(d>=23))|((m==7)&(d<=22))).astype(int)
    df['in_yearend'] = (((m==11)&(d>=18))|(m==12)|((m==1)&(d<=2))).astype(int)
    
    # V17 Feature Engineering: Distinct Holiday Flags for better tree splits
    df['is_liberation_day'] = ((m==4)&(d==30)).astype(int)
    df['is_labor_day'] = ((m==5)&(d==1)).astype(int)
    df['is_national_day'] = ((m==9)&(d==2)).astype(int)
    
    df['days_to_end'] = df.Date.dt.days_in_month - d
    df['is_tet_season'] = (((m==1)&(d>=15)) | ((m==2)&(d<=15))).astype(int)
    df['is_double_day'] = (m == d).astype(int)
    
    return df.drop(columns='Date')

def make_features(dates, history):
    df = pd.DataFrame({'Date': dates})
    df['month'], df['day'], df['dow'], df['doy'] = df.Date.dt.month, df.Date.dt.day, df.Date.dt.dayofweek, df.Date.dt.dayofyear
    
    df['enc_month'] = df['month'].map(target_enc_month)
    df['enc_dow'] = df['dow'].map(target_enc_dow)
    df['enc_day'] = df['day'].map(target_enc_day)

    for k in [1, 2, 3]:
        df[f'sin_y_{k}'] = np.sin(2*np.pi*k*df.doy/365.25)
        df[f'cos_y_{k}'] = np.cos(2*np.pi*k*df.doy/365.25)
    
    for lag in [1, 7, 30, 365]:
        df[f'lag_{lag}'] = (df.Date - pd.Timedelta(days=lag)).map(history)
        
    df['lag_diff_1_7'] = df['lag_1'] - df['lag_7']
    
    h_sorted = history.sort_index()
    # V17 Feature Engineering: Added roll_mean_3_lag_1
    for w in [3, 7, 30]:
        df[f'roll_mean_{w}'] = (df.Date - pd.Timedelta(days=1)).map(h_sorted.rolling(w).mean())
        
    df['momentum_7_30'] = df['roll_mean_7'] / (df['roll_mean_30'] + 1e-5)
    
    df = pd.concat([df, get_promo_features(df.Date)], axis=1)
    df = df.merge(wt, on='Date', how='left').fillna(0)
    return df.drop(columns='Date')

history_init = sales.set_index('Date')['Revenue']
X = make_features(sales['Date'], history_init)
y = sales['Revenue']


In [ ]:
# Optuna Tuning with Deep Hyperparameter Search (V17)
tscv = TimeSeriesSplit(n_splits=5)

def lgbm_mae_objective(trial):
    params = {
        'objective': 'mae', 'verbosity': -1, 'n_estimators': 800, 'random_state': 42,
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True)
    }
    scores = []
    for tr_idx, va_idx in tscv.split(X):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[va_idx], y.iloc[va_idx])], callbacks=[lgb.early_stopping(30, verbose=False)])
        scores.append(mean_absolute_error(y.iloc[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(scores)

def lgbm_rmse_objective(trial):
    params = {
        'objective': 'rmse', 'verbosity': -1, 'n_estimators': 800, 'random_state': 42,
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True)
    }
    scores = []
    for tr_idx, va_idx in tscv.split(X):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[va_idx], y.iloc[va_idx])], callbacks=[lgb.early_stopping(30, verbose=False)])
        scores.append(mean_absolute_error(y.iloc[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(scores)

def cat_objective(trial):
    params = {
        'loss_function': 'MAE', 'iterations': 800, 'verbose': 0, 'random_seed': 42,
        'depth': trial.suggest_int('depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-4, 30.0, log=True)
    }
    scores = []
    for tr_idx, va_idx in tscv.split(X):
        m = CatBoostRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[va_idx], y.iloc[va_idx])], early_stopping_rounds=30)
        scores.append(mean_absolute_error(y.iloc[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(scores)

def xgb_objective(trial):
    params = {
        'objective': 'reg:absoluteerror', 'n_estimators': 800, 'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'alpha': trial.suggest_float('alpha', 1e-4, 20.0, log=True),
        'lambda': trial.suggest_float('lambda', 1e-4, 20.0, log=True)
    }
    scores = []
    for tr_idx, va_idx in tscv.split(X):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[va_idx], y.iloc[va_idx])], verbose=False)
        scores.append(mean_absolute_error(y.iloc[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(scores)


In [ ]:
print("Tuning LightGBM (MAE) - 50 Trials...")
study_lgb_mae = optuna.create_study(direction='minimize')
study_lgb_mae.optimize(lgbm_mae_objective, n_trials=50)

print("Tuning LightGBM (RMSE Spike Booster) - 50 Trials...")
study_lgb_rmse = optuna.create_study(direction='minimize')
study_lgb_rmse.optimize(lgbm_rmse_objective, n_trials=50)

print("Tuning CatBoost - 30 Trials...")
study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(cat_objective, n_trials=30)

print("Tuning XGBoost - 30 Trials...")
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(xgb_objective, n_trials=30)


In [ ]:
# Generate OOF predictions for Objective Diversity Blending
oof_lgb_mae = np.zeros(len(X))
oof_lgb_rmse = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, va_idx in kf.split(X):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
    m1 = lgb.LGBMRegressor(**study_lgb_mae.best_params, objective='mae', n_estimators=800, verbosity=-1, random_state=42).fit(X_tr, y_tr)
    oof_lgb_mae[va_idx] = m1.predict(X_va)
    
    m2 = lgb.LGBMRegressor(**study_lgb_rmse.best_params, objective='rmse', n_estimators=800, verbosity=-1, random_state=42).fit(X_tr, y_tr)
    oof_lgb_rmse[va_idx] = m2.predict(X_va)
    
    m3 = CatBoostRegressor(**study_cat.best_params, loss_function='MAE', iterations=800, verbose=0, random_seed=42).fit(X_tr, y_tr)
    oof_cat[va_idx] = m3.predict(X_va)
    
    m4 = xgb.XGBRegressor(**study_xgb.best_params, objective='reg:absoluteerror', n_estimators=800, random_state=42).fit(X_tr, y_tr)
    oof_xgb[va_idx] = m4.predict(X_va)


In [ ]:
# Optuna Blending Weights Optimization
def weight_objective(trial):
    w1 = trial.suggest_float('w1', 0.0, 1.0)
    w2 = trial.suggest_float('w2', 0.0, 1.0)
    w3 = trial.suggest_float('w3', 0.0, 1.0)
    w4 = trial.suggest_float('w4', 0.0, 1.0)
    total = w1 + w2 + w3 + w4
    if total == 0:
        return float('inf')
    w1, w2, w3, w4 = w1/total, w2/total, w3/total, w4/total
    
    blend_preds = w1*oof_lgb_mae + w2*oof_lgb_rmse + w3*oof_cat + w4*oof_xgb
    return mean_absolute_error(y, blend_preds)

study_weights = optuna.create_study(direction='minimize')
study_weights.optimize(weight_objective, n_trials=100)

w1_raw = study_weights.best_params['w1']
w2_raw = study_weights.best_params['w2']
w3_raw = study_weights.best_params['w3']
w4_raw = study_weights.best_params['w4']
total = w1_raw + w2_raw + w3_raw + w4_raw
best_w = (w1_raw/total, w2_raw/total, w3_raw/total, w4_raw/total)
print(f"Optimal Blending Weights: LGB_MAE={best_w[0]:.3f}, LGB_RMSE={best_w[1]:.3f}, CAT={best_w[2]:.3f}, XGB={best_w[3]:.3f}")


In [ ]:
# Final Inference: V17 with 15-Seed Averaging
test_dates = sample_sub['Date'].sort_values().tolist()
w_lgb_mae, w_lgb_rmse, w_cat, w_xgb = best_w
final_blend_preds = np.zeros(len(test_dates))

# 15 seeds for variance reduction
seeds = [42, 2026, 999, 123, 777, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
num_seeds = len(seeds)

for s in seeds:
    print(f"\n--- Training and Predicting with Seed {s} ---")
    
    final_lgb_mae = lgb.LGBMRegressor(**study_lgb_mae.best_params, objective='mae', n_estimators=1000, verbosity=-1, random_state=s).fit(X, y)
    final_lgb_rmse = lgb.LGBMRegressor(**study_lgb_rmse.best_params, objective='rmse', n_estimators=1000, verbosity=-1, random_state=s).fit(X, y)
    final_cat = CatBoostRegressor(**study_cat.best_params, loss_function='MAE', iterations=1000, verbose=0, random_seed=s).fit(X, y)
    final_xgb = xgb.XGBRegressor(**study_xgb.best_params, objective='reg:absoluteerror', n_estimators=1000, random_state=s).fit(X, y)
    
    rolling_history = history_init.copy()
    preds_s = []
    
    for d in test_dates:
        row = make_features(pd.Series([d]), rolling_history)
        
        p1 = final_lgb_mae.predict(row)[0]
        p2 = final_lgb_rmse.predict(row)[0]
        p3 = final_cat.predict(row)[0]
        p4 = final_xgb.predict(row)[0]
        
        p_final = w_lgb_mae*p1 + w_lgb_rmse*p2 + w_cat*p3 + w_xgb*p4
        p_final = np.clip(p_final, 0, None)
        
        rolling_history[d] = p_final
        preds_s.append(p_final)
        
    # Add to the global average
    final_blend_preds += np.array(preds_s) / num_seeds
    
    # Memory cleanup
    del final_lgb_mae, final_lgb_rmse, final_cat, final_xgb
    gc.collect()

sub = sample_sub.copy()
sub['Revenue'] = final_blend_preds
sub['COGS'] = np.array(final_blend_preds) * (sales['COGS'].sum()/sales['Revenue'].sum())
sub['Date'] = sub['Date'].dt.strftime('%Y-%m-%d')
sub.to_csv('submission_v17.csv', index=False)
print("\nDone! V17 (15-Seed + Enhanced Optuna + roll_mean_3 + distinct holidays) predictions saved to submission_v17.csv")
sub.head()
